# Extension 4 — State-level rural CHC context and sector association

Run this notebook top to bottom using the **frozen** `data/processed/df_model_v2.parquet`. The analysis is a contextual association, not a causal effect of CHC staffing or changing facilities. Results are generated by the included `analysis.py`; no precomputed outputs are loaded. This run cross-fits two five-fold models and then runs 500 paired PSU bootstrap replicates, so the model-fitting cell may take time.


## 1. Locate the extension code and check input paths


In [ ]:
import os, sys
from pathlib import Path

root = None
for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (p / "extensions_work" / "04_health_system_context" / "analysis.py").exists():
        root = p
        break
if root is None:
    raise FileNotFoundError("Open this notebook from inside the IPD repository; analysis.py was not found.")
extension_dir = root / "extensions_work" / "04_health_system_context"
sys.path.insert(0, str(extension_dir))
from analysis import run_extension4, OUTPUTS
v2_path = Path(os.environ.get("IPD_EXT04_V2_PATH", root / "data" / "processed" / "df_model_v2.parquet"))
print("Repository:", root)
print("Frozen V2:", v2_path)
print("Extension:", extension_dir)
print("Input exists:", v2_path.is_file())


## 2. Recompute results and write outputs

Before running, move any previous CSV, JSON or PNG files out of `outputs/` to a backup folder. This cell refuses to reuse or overwrite those files. It reads the real frozen V2 Parquet and generates all outputs from scratch.


In [ ]:
modifier_results, metadata = run_extension4()
display(modifier_results)
print("Delta (high - low), pp:", metadata["results"]["delta_high_minus_low_pct"])
print("Paired PSU bootstrap interval:", metadata["results"]["delta_ci_pct"])


## 3. Check generated outputs


In [ ]:
import pandas as pd
out = extension_dir / "outputs"
missing = [f for f in OUTPUTS if not (out / f).is_file()]
assert not missing, f"Missing generated files: {missing}"
desc = pd.read_csv(out / "context_descriptives.csv")
assert len(desc) == 4 and set(desc.context_group) == {"low", "high"}
assert (desc.groupby("context_group").weighted_share_pct.sum() - 100).abs().max() < 1e-8
influence = pd.read_csv(out / "context_leave_one_state_out.csv")
boot = pd.read_csv(out / "context_bootstrap_results.csv")
assert len(boot) == 500 and len(influence) == 34
assert (boot.rd_high_pct - boot.rd_low_pct - boot.delta_high_minus_low_pct).abs().max() < 1e-8
print("Four descriptive rows, 500 joint bootstrap draws and 34 leave-one-state-out checks passed.")
print("State omission Delta range:", influence.delta_high_minus_low_pct.min(), influence.delta_high_minus_low_pct.max())
print("Generated files:")
for f in OUTPUTS: print(" -", f)
display(desc)
